# Aufgabe 4b: Lineare Single-Cell-SVM

Dieses Notebook implementiert die lineare Single-Cell-SVM mit spenderweiser innerer Modellauswahl und äußerer Testauswertung. Der voreingestellte Smoke-Modus verwendet `gated_NK`; der vollständige Hauptvergleich verwendet `gated_alive`. Gate und Modus des tatsächlich ausgeführten Laufs stehen in der Konfigurationsausgabe.


## 1. Konfiguration

`gated_NK` dient als schneller technischer Test. Im Full-Modus wird der Hauptvergleich mit `gated_alive` und allen 100 gemeinsamen Spendersplits ausgeführt.


In [1]:
# Zweck: Laufmodus, reproduzierbare Parameter und Ein-/Ausgabepfade der SVM festlegen.
from pathlib import Path
import csv
import os
import warnings

import flowkit as fk
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    auc,
    average_precision_score,
    balanced_accuracy_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

RUN_MODE = os.environ.get("TASK4_RUN_MODE", "smoke")
RUN_TRAINING = os.environ.get("TASK4_RUN_TRAINING", "1") == "1"
if RUN_MODE == "smoke":
    GATE = "gated_NK"
    ANALYSIS_SPLIT_IDS = [0, 1]
    TRAIN_CELLS_PER_DONOR = 2_000
elif RUN_MODE == "full":
    GATE = "gated_alive"
    ANALYSIS_SPLIT_IDS = list(range(100))
    TRAIN_CELLS_PER_DONOR = 10_000
else:
    raise ValueError(f"Unbekannter RUN_MODE: {RUN_MODE}")
SPLIT_LIMIT = int(os.environ.get("TASK4_SPLIT_LIMIT", "0"))
if SPLIT_LIMIT > 0:
    ANALYSIS_SPLIT_IDS = ANALYSIS_SPLIT_IDS[:SPLIT_LIMIT]
FORCE_SPLIT_IDS = {
    int(value)
    for value in os.environ.get("TASK4_FORCE_SPLITS", "").split(",")
    if value.strip()
}
GATE_SUFFIXES = {"gated_NK": "_NK", "gated_alive": "_alive"}
EXPECTED_EVENT_COUNTS = {"gated_NK": 261_593, "gated_alive": 3_438_750}
if GATE not in GATE_SUFFIXES:
    raise ValueError(f"Unbekannter Gate: {GATE}")
GATE_SUFFIX = GATE_SUFFIXES[GATE]
COFACTOR = 5.0
TECHNICAL_SPLIT_IDS = [0, 1]
TECHNICAL_INNER_FOLD = 0
TECHNICAL_CELLS_PER_DONOR = 2_000
SAMPLING_SEED_OFFSET = 20_000
SVM_C_VALUES = [0.01, 0.1, 1.0]
TOP_FRACTION = 0.01
SVM_MAX_ITERATIONS = 10_000
SVM_TOLERANCE = 1e-4

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "NK_cell_dataset").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "NK_cell_dataset" / "NK_cell_dataset"
FCS_DIR = DATA_ROOT / "NK_cell_dataset" / GATE
LABELS_PATH = DATA_ROOT / "NK_fcs_samples_with_labels.csv"
MARKERS_PATH = DATA_ROOT / "NK_markers.csv"
SPLITS_PATH = PROJECT_ROOT / "results" / "tables" / "task4_donor_splits.csv"
SVM_PREDICTIONS_PATH = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / f"task4_svm_predictions_{GATE}_{RUN_MODE}.csv"
)
SVM_SELECTION_PATH = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / f"task4_svm_selection_{GATE}_{RUN_MODE}.csv"
)

SVM_MODELS_PATH = SVM_PREDICTIONS_PATH.with_name(
    f"task4_svm_models_{GATE}_{RUN_MODE}.csv"
)
CONFIG_PATH = SVM_PREDICTIONS_PATH.with_suffix(".config.json")

import sys
from importlib.metadata import version
sys.path.insert(0, str(PROJECT_ROOT))
from src.task4_artifacts import (
    make_run_config, check_run_config, write_run_config,
    validate_prediction_splits, validate_parameter_table,
)

for required_path in (FCS_DIR, LABELS_PATH, MARKERS_PATH, SPLITS_PATH):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Erwarteter Pfad fehlt: {required_path}. "
            "Gegebenenfalls zuerst 04a_data_qc_and_splits.ipynb ausführen."
        )

print(
    f"Modus: {RUN_MODE}; Gate: {GATE}; Analysesplits: {len(ANALYSIS_SPLIT_IDS)}; "
    f"Training: {'aktiv' if RUN_TRAINING else 'deaktiviert'}"
)

Modus: full; Gate: gated_alive; Analysesplits: 100; Training: aktiv


## 2. Marker, Labels und gemeinsame Splits laden

In [2]:
# Zweck: Marker, Labels, FCS-Dateien und die für alle Methoden gemeinsamen Splits verknüpfen.
with MARKERS_PATH.open(newline="", encoding="utf-8-sig") as stream:
    markers = next(csv.reader(stream))

label_table = pd.read_csv(LABELS_PATH)
label_table["donor_id"] = label_table["fcs_filename"].str.replace(
    r"_NK\.fcs$", "", regex=True
)
label_table["label"] = label_table["label"].astype(int)

fcs_paths = sorted(FCS_DIR.glob("*.fcs"))
fcs_table = pd.DataFrame(
    {
        "donor_id": [path.stem.removesuffix(GATE_SUFFIX) for path in fcs_paths],
        "fcs_path": fcs_paths,
    }
)
sample_table = (
    label_table[["donor_id", "label"]]
    .merge(fcs_table, on="donor_id", validate="one_to_one")
    .sort_values("donor_id")
    .reset_index(drop=True)
)
donor_splits = pd.read_csv(SPLITS_PATH)

assert len(markers) == 37
assert len(fcs_paths) == 20
assert len(sample_table) == 20
assert set(sample_table["donor_id"]) == set(donor_splits["donor_id"])
split_labels = donor_splits[["donor_id", "label"]].drop_duplicates()
expected_labels = sample_table[["donor_id", "label"]]
assert split_labels.merge(
    expected_labels, on=["donor_id", "label"], validate="one_to_one"
).shape[0] == len(expected_labels)
assert set(TECHNICAL_SPLIT_IDS).issubset(set(donor_splits["split_id"]))
assert donor_splits.groupby(["split_id", "donor_id"]).size().eq(1).all()

display(sample_table[["donor_id", "label"]])

,donor_id,label
0,a_001,1
1,a_002,1
2,a_003,0
3,a_004,0
4,a_005,1
5,a_006,0
6,a_007,1
7,a_009,0
8,a_010,0
9,a_011,0


## 3. Transformierte Daten spenderweise laden

Die Daten bleiben als getrennte Arrays pro Spender erhalten. Dadurch kann ein Spender nur vollständig einer Trainings-, Validierungs- oder Testmenge zugeordnet werden.

In [3]:
# Zweck: Jeden Spender read-only laden, auf 37 Marker begrenzen und fest transformieren.
def load_transformed_fcs(path: Path, selected_markers: list[str]) -> np.ndarray:
    """Eine FCS-Datei als ArcSinh-transformierte Zell-mal-Marker-Matrix laden.

    Auswahl und Reihenfolge folgen ``selected_markers``; nicht-endliche Werte
    führen sofort zu einem Fehler statt später das Modell zu verfälschen.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"FCS file .* reported incorrect data offset.*",
        )
        sample = fk.Sample(str(path), ignore_offset_error=True)

    frame = sample.as_dataframe(source="raw")
    short_names = pd.Index(sample.pns_labels, name="marker")
    if not short_names.is_unique:
        raise ValueError(f"Doppelte FCS-Kurznamen in {path.name}.")
    frame.columns = short_names
    missing_markers = [marker for marker in selected_markers if marker not in frame.columns]
    if missing_markers:
        raise ValueError(f"Fehlende Marker in {path.name}: {missing_markers}")

    raw_values = frame.loc[:, selected_markers].to_numpy(dtype=np.float32, copy=True)
    transformed_values = np.arcsinh(raw_values / COFACTOR)
    if not np.isfinite(transformed_values).all():
        raise ValueError(f"Nicht-endliche Werte in {path.name}.")
    return transformed_values


# Getrennte Arrays halten den Spender als kleinste teilbare Beobachtungseinheit sichtbar.
data_by_donor = {
    row.donor_id: load_transformed_fcs(row.fcs_path, markers)
    for row in sample_table.itertuples(index=False)
}
label_by_donor = sample_table.set_index("donor_id")["label"].to_dict()

loaded_overview = pd.DataFrame(
    [
        {
            "donor_id": donor_id,
            "label": label_by_donor[donor_id],
            "cells": len(values),
            "markers": values.shape[1],
        }
        for donor_id, values in data_by_donor.items()
    ]
)
assert loaded_overview["cells"].sum() == EXPECTED_EVENT_COUNTS[GATE]
assert loaded_overview["markers"].eq(37).all()
display(loaded_overview)

,donor_id,label,cells,markers
0,a_001,1,82324,37
1,a_002,1,108267,37
2,a_003,0,97529,37
3,a_004,0,140687,37
4,a_005,1,155335,37
5,a_006,0,90075,37
6,a_007,1,104805,37
7,a_009,0,216632,37
8,a_010,0,122209,37
9,a_011,0,258461,37


## 4. Spenderbalanciertes Sampling und fold-sichere Skalierung

Für den technischen Test werden nur aus den Trainingsspendern gleich viele Zellen ohne Zurücklegen gezogen. Der `StandardScaler` sieht ausschließlich diese Zellen der inneren Trainingsspender. Innere Validierungs- und äußere Testspender werden vollständig und mit unveränderten Trainingsparametern transformiert.

In [4]:
# Zweck: Zellmatrizen für einen Fold bilden, ohne Spendergrenzen oder Testdaten zu verletzen.
def stack_sampled_donors(
    donor_ids: list[str],
    cells_per_donor: int,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Aus jedem genannten Spender gleich viele Zellen ohne Zurücklegen ziehen.

    Zurückgegeben werden Markerwerte, schwache Zelllabels und die zugehörigen
    Spender-IDs. Die Gleichverteilung verhindert eine Gewichtung nach Zellzahl.
    """
    sampled_values = []
    sampled_labels = []
    sampled_donor_ids = []

    for donor_id in sorted(donor_ids):
        donor_values = data_by_donor[donor_id]
        if len(donor_values) < cells_per_donor:
            raise ValueError(
                f"{donor_id} besitzt nur {len(donor_values)} Zellen; "
                f"angefordert sind {cells_per_donor}."
            )
        indices = rng.choice(len(donor_values), size=cells_per_donor, replace=False)
        sampled_values.append(donor_values[indices])
        sampled_labels.append(
            np.full(cells_per_donor, label_by_donor[donor_id], dtype=np.int8)
        )
        sampled_donor_ids.append(
            np.full(cells_per_donor, donor_id, dtype=object)
        )

    return (
        np.concatenate(sampled_values),
        np.concatenate(sampled_labels),
        np.concatenate(sampled_donor_ids),
    )


def stack_all_donors(
    donor_ids: list[str],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Alle Zellen der genannten Spender mitsamt Label und Spender-ID stapeln."""
    values = [data_by_donor[donor_id] for donor_id in sorted(donor_ids)]
    labels = [
        np.full(len(data_by_donor[donor_id]), label_by_donor[donor_id], dtype=np.int8)
        for donor_id in sorted(donor_ids)
    ]
    cell_donor_ids = [
        np.full(len(data_by_donor[donor_id]), donor_id, dtype=object)
        for donor_id in sorted(donor_ids)
    ]
    return np.concatenate(values), np.concatenate(labels), np.concatenate(cell_donor_ids)


def prepare_inner_fold(
    split_id: int,
    validation_fold: int,
    cells_per_donor: int,
) -> dict[str, object]:
    """Einen inneren Train-/Validierungsfold leakage-sicher vorbereiten.

    Der Scaler wird ausschließlich auf den gezogenen Zellen der inneren
    Trainingsspender gefittet. Äußere Testspender bleiben vollständig unberührt.
    """
    split = donor_splits.loc[donor_splits["split_id"] == split_id].copy()
    inner_train_ids = split.loc[
        (split["outer_partition"] == "train")
        & (split["inner_fold"] != validation_fold),
        "donor_id",
    ].tolist()
    inner_validation_ids = split.loc[
        (split["outer_partition"] == "train")
        & (split["inner_fold"] == validation_fold),
        "donor_id",
    ].tolist()
    outer_test_ids = split.loc[
        split["outer_partition"] == "test", "donor_id"
    ].tolist()

    donor_sets = [set(inner_train_ids), set(inner_validation_ids), set(outer_test_ids)]
    assert not donor_sets[0] & donor_sets[1]
    assert not donor_sets[0] & donor_sets[2]
    assert not donor_sets[1] & donor_sets[2]
    assert set.union(*donor_sets) == set(data_by_donor)

    split_seed = int(split["split_seed"].iloc[0])
    rng = np.random.default_rng(
        split_seed + SAMPLING_SEED_OFFSET + validation_fold
    )
    train = stack_sampled_donors(inner_train_ids, cells_per_donor, rng)
    validation = stack_all_donors(inner_validation_ids)

    # Wichtigster Leakage-Schutz: fit nur auf ``train[0]``, danach nur transformieren.
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train[0])
    validation_scaled = scaler.transform(validation[0])

    assert np.isfinite(train_scaled).all()
    assert np.isfinite(validation_scaled).all()
    assert np.max(np.abs(train_scaled.mean(axis=0))) < 1e-5

    return {
        "split_id": split_id,
        "validation_fold": validation_fold,
        "train": (train_scaled, train[1], train[2]),
        "validation": (validation_scaled, validation[1], validation[2]),
        "outer_test_ids": outer_test_ids,
        "scaler": scaler,
    }


## 5. Technischer Test auf zwei Splits

In [5]:
# Zweck: Datenfluss und Trennung auf zwei kleinen Folds prüfen, bevor Modelle trainiert werden.
technical_fold_data = [
    prepare_inner_fold(
        split_id=split_id,
        validation_fold=TECHNICAL_INNER_FOLD,
        cells_per_donor=TECHNICAL_CELLS_PER_DONOR,
    )
    for split_id in TECHNICAL_SPLIT_IDS
]

technical_records = []
for fold_data in technical_fold_data:
    test_unscaled = stack_all_donors(fold_data["outer_test_ids"])
    test = (
        fold_data["scaler"].transform(test_unscaled[0]),
        test_unscaled[1],
        test_unscaled[2],
    )
    partitions = {
        "train": fold_data["train"],
        "validation": fold_data["validation"],
        "test": test,
    }
    for partition, (values, labels, donor_ids) in partitions.items():
        technical_records.append(
            {
                "split_id": fold_data["split_id"],
                "partition": partition,
                "donors": len(np.unique(donor_ids)),
                "cells": len(values),
                "CMV− cells": int((labels == 0).sum()),
                "CMV+ cells": int((labels == 1).sum()),
                "finite": bool(np.isfinite(values).all()),
            }
        )

technical_summary = pd.DataFrame(technical_records)
assert technical_summary["finite"].all()
assert technical_summary.loc[technical_summary["partition"] == "test", "donors"].eq(6).all()
display(technical_summary)

,split_id,partition,donors,cells,CMV− cells,CMV+ cells,finite
0,0,train,9,18000,10000,8000,True
1,0,validation,5,834052,355990,478062,True
2,0,test,6,1024234,604252,419982,True
3,1,train,9,18000,10000,8000,True
4,1,validation,5,1041239,438628,602611,True
5,1,test,6,908197,544939,363258,True


## Abschluss der technischen Datenflussprüfung

Die Prüfung verwendet zwei feste Splits der aktiven Gate-Stufe. Sampling, innere Validierung, äußerer Test und ausschließlich auf Trainingsspendern gefittete Standardisierung sind damit kontrolliert.


## 6. Lineare Single-Cell-SVM

Jede Trainingszelle erhält das Label ihres Spenders. Für jedes `C` wird die Leistung in den drei inneren Validierungsfolds ausschließlich auf Spenderebene bewertet. Die kontinuierlichen Zell-Scores werden pro Spender als Mittelwert der höchsten 1 % aggregiert.

Die AUC bewertet die Rangfolge der Spender. Unterschiedliche C-Werte können daher
verschiedene Zell- und Spenderscores, aber dieselbe AUC liefern. Bei gleichen
mittleren Validierungs-AUCs wird vorab festgelegt das kleinste C gewählt; daraus
folgt keine nachgewiesene Überlegenheit dieses C-Werts. Die Scores nach
Top-1-%-Aggregation sind keine Wahrscheinlichkeiten und benötigen eine eigene
Spenderschwelle.

Nach Auswahl von `C` wird aus den zusammengeführten inneren Out-of-fold-Spendervorhersagen ein Schwellenwert nach dem Youden-Kriterium bestimmt. Anschließend wird die SVM auf allen 14 äußeren Trainingsspendern neu gefittet und genau einmal auf den sechs äußeren Testspendern ausgewertet.

In [6]:
# Zweck: Zell-Scores zu Spender-Scores verdichten und die SVM verschachtelt auswählen.
def aggregate_top_fraction(
    cell_scores: np.ndarray,
    cell_donor_ids: np.ndarray,
    top_fraction: float = TOP_FRACTION,
) -> pd.DataFrame:
    """Je Spender den Mittelwert seiner höchsten Zell-Scores berechnen.

    Die Top-1%-Aggregation ist auf seltene informative Populationen ausgerichtet;
    jeder Spender liefert unabhängig von seiner Zellzahl genau einen Score.
    """
    if not 0 < top_fraction <= 1:
        raise ValueError("top_fraction muss im Intervall (0, 1] liegen.")

    records = []
    for donor_id in sorted(np.unique(cell_donor_ids)):
        donor_scores = cell_scores[cell_donor_ids == donor_id]
        top_count = max(1, int(np.ceil(top_fraction * len(donor_scores))))
        # ``partition`` findet die größten Werte ohne die gesamte Folge zu sortieren.
        top_scores = np.partition(donor_scores, -top_count)[-top_count:]
        records.append(
            {
                "donor_id": donor_id,
                "y_true": label_by_donor[donor_id],
                "score": float(top_scores.mean()),
                "top_cell_count": top_count,
            }
        )
    return pd.DataFrame(records)


def select_youden_threshold(y_true: np.ndarray, scores: np.ndarray) -> float:
    """Den Schwellenwert mit maximaler Sensitivität plus Spezifität auswählen."""
    false_positive_rate, true_positive_rate, thresholds = roc_curve(y_true, scores)
    youden_index = true_positive_rate - false_positive_rate
    finite = np.isfinite(thresholds)
    if not finite.any():
        raise ValueError("Kein endlicher Schwellenwert verfügbar.")
    finite_indices = np.flatnonzero(finite)
    best_index = finite_indices[int(np.argmax(youden_index[finite]))]
    return float(thresholds[best_index])


def trapezoidal_pr_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    """Die Fläche unter der Precision-Recall-Kurve trapezoidal integrieren."""
    precision, recall, _ = precision_recall_curve(y_true, scores)
    return float(auc(recall, precision))


def select_svm_hyperparameters(
    split_id: int,
    cells_per_donor: int,
) -> tuple[float, float, pd.DataFrame]:
    """``C`` und Entscheidungsschwelle nur per innerer Spender-CV bestimmen.

    Ergebnis sind das beste ``C``, der aus Out-of-fold-Scores gelernte
    Schwellenwert und alle Kandidatenmetriken zur Dokumentation.
    """
    candidate_records = []
    out_of_fold_by_c = {}

    for c_value in SVM_C_VALUES:
        fold_predictions = []
        for validation_fold in range(3):
            fold_data = prepare_inner_fold(
                split_id=split_id,
                validation_fold=validation_fold,
                cells_per_donor=cells_per_donor,
            )
            train_values, train_labels, _ = fold_data["train"]
            validation_values, _, validation_donors = fold_data["validation"]
            split_seed = int(
                donor_splits.loc[donor_splits["split_id"] == split_id, "split_seed"].iloc[0]
            )
            classifier = LinearSVC(
                C=c_value,
                dual="auto",
                max_iter=SVM_MAX_ITERATIONS,
                tol=SVM_TOLERANCE,
                random_state=split_seed + validation_fold,
            )
            classifier.fit(train_values, train_labels)
            validation_scores = classifier.decision_function(validation_values)
            donor_predictions = aggregate_top_fraction(
                validation_scores, validation_donors
            )
            donor_predictions["inner_fold"] = validation_fold
            fold_predictions.append(donor_predictions)
            candidate_records.append(
                {
                    "split_id": split_id,
                    "C": c_value,
                    "inner_fold": validation_fold,
                    "validation_roc_auc": roc_auc_score(
                        donor_predictions["y_true"], donor_predictions["score"]
                    ),
                }
            )

        # Jeder Trainingsspender ist hier genau einmal eine ungesehene Validierung.
        out_of_fold_by_c[c_value] = pd.concat(fold_predictions, ignore_index=True)

    candidate_results = pd.DataFrame(candidate_records)
    mean_auc_by_c = (
        candidate_results.groupby("C")["validation_roc_auc"].mean().sort_index()
    )
    best_c = float(
        mean_auc_by_c.reset_index()
        .sort_values(["validation_roc_auc", "C"], ascending=[False, True])
        .iloc[0]["C"]
    )
    best_out_of_fold = out_of_fold_by_c[best_c]
    threshold = select_youden_threshold(
        best_out_of_fold["y_true"].to_numpy(),
        best_out_of_fold["score"].to_numpy(),
    )
    return best_c, threshold, candidate_results


def fit_and_predict_outer_split(
    split_id: int,
    cells_per_donor: int,
    best_c: float,
    threshold: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Finale SVM auf 14 Trainingsspendern fitten und 6 Testspender bewerten."""
    split = donor_splits.loc[donor_splits["split_id"] == split_id]
    train_ids = split.loc[split["outer_partition"] == "train", "donor_id"].tolist()
    test_ids = split.loc[split["outer_partition"] == "test", "donor_id"].tolist()
    assert not set(train_ids) & set(test_ids)

    split_seed = int(split["split_seed"].iloc[0])
    rng = np.random.default_rng(split_seed + SAMPLING_SEED_OFFSET + 100)
    train_values, train_labels, _ = stack_sampled_donors(
        train_ids, cells_per_donor, rng
    )
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_values)
    classifier = LinearSVC(
        C=best_c,
        dual="auto",
        max_iter=SVM_MAX_ITERATIONS,
                tol=SVM_TOLERANCE,
        random_state=split_seed,
    )
    classifier.fit(train_scaled, train_labels)

    # Erst nach abgeschlossenem Fit werden die äußeren Testzellen transformiert.
    test_records = []
    for donor_id in sorted(test_ids):
        donor_values = scaler.transform(data_by_donor[donor_id])
        donor_cell_scores = classifier.decision_function(donor_values)
        top_count = max(1, int(np.ceil(TOP_FRACTION * len(donor_cell_scores))))
        top_scores = np.partition(donor_cell_scores, -top_count)[-top_count:]
        donor_score = float(top_scores.mean())
        test_records.append(
            {
                "method": "linear_single_cell_svm",
                "gate": GATE,
                "run_mode": RUN_MODE,
                "split_id": split_id,
                "split_seed": split_seed,
                "donor_id": donor_id,
                "y_true": label_by_donor[donor_id],
                "score": donor_score,
                "decision_threshold": threshold,
                "y_pred": int(donor_score >= threshold),
                "best_C": best_c,
                "top_fraction": TOP_FRACTION,
                "top_cell_count": top_count,
                "train_cells_per_donor": cells_per_donor,
            }
        )
    # Die vollständige lineare Regel und der Trainings-Scaler reichen für Aufgabe 5.
    model_parameters = pd.DataFrame({
        "split_id": split_id, "split_seed": split_seed,
        "gate": GATE, "run_mode": RUN_MODE, "marker": markers,
        "weight": classifier.coef_[0], "intercept": float(classifier.intercept_[0]),
        "scaler_mean": scaler.mean_, "scaler_scale": scaler.scale_,
        "cofactor": COFACTOR, "top_fraction": TOP_FRACTION,
        "best_C": best_c, "decision_threshold": threshold,
        "negative_class": int(classifier.classes_[0]),
        "positive_class": int(classifier.classes_[1]),
    })
    return pd.DataFrame(test_records), model_parameters

## 7. SVM ausführen oder vorhandene Ergebnisse laden

Im voreingestellten Smoke-Modus werden zwei Splits und 2.000 Trainingszellen je Spender verwendet. Im Full-Modus laufen die 100 gemeinsamen Splits auf `gated_alive` mit 10.000 Trainingszellen je Spender. Ergebnisse, Gewichte, Intercept und Trainings-Scaler werden nach jedem Split
gespeichert. Eine Fortsetzung setzt passende Parameter, identische Eingabedateien
und vollständige Artefakte voraus. Altbestände ohne Konfigurationsnachweis werden
hier abgewiesen; der historische Vergleich in `04e` bleibt möglich.

In [7]:
# Zweck: Nur passende, vollständige Zwischenstände laden und Modellparameter mitspeichern.
run_config = make_run_config(
    parameters={
        "method": "linear_single_cell_svm", "gate": GATE, "run_mode": RUN_MODE,
        "C_values": SVM_C_VALUES, "train_cells_per_donor": TRAIN_CELLS_PER_DONOR,
        "cofactor": COFACTOR, "top_fraction": TOP_FRACTION,
        "sampling_seed_offset": SAMPLING_SEED_OFFSET,
        "max_iterations": SVM_MAX_ITERATIONS, "tolerance": SVM_TOLERANCE,
        "loss": "squared_hinge", "penalty": "l2", "dual": "auto",
        "scaling": "training_donors_equal_cells", "threshold": "inner_oof_youden",
        "selection": "mean_inner_auc_then_smallest_C",
        "numpy": version("numpy"), "scikit_learn": version("scikit-learn"),
        "flowkit": version("flowkit"), "model_format": "marker_parameters_v1",
    },
    input_paths=[SPLITS_PATH, LABELS_PATH, MARKERS_PATH, *fcs_paths],
    notebook_path=PROJECT_ROOT / "notebooks" / "04b_svm.ipynb",
    code_cells=[4, 6, 8, 13],
)
required_model_columns = ['weight', 'intercept', 'scaler_mean', 'scaler_scale', 'cofactor', 'top_fraction', 'best_C', 'decision_threshold', 'negative_class', 'positive_class']
has_checkpoint = check_run_config(
    CONFIG_PATH, run_config,
    [SVM_PREDICTIONS_PATH, SVM_SELECTION_PATH, SVM_MODELS_PATH], RUN_TRAINING,
)
svm_predictions_by_split, svm_selection_by_split, svm_models_by_split = [], [], []
completed_split_ids = set()
if has_checkpoint:
    previous_predictions = pd.read_csv(SVM_PREDICTIONS_PATH)
    previous_selection = pd.read_csv(SVM_SELECTION_PATH)
    previous_models = pd.read_csv(SVM_MODELS_PATH, float_precision="round_trip")
    validate_prediction_splits(previous_predictions, donor_splits)
    validate_parameter_table(previous_models, previous_predictions, markers, ["split_id"], required_model_columns)
    expected_candidates = pd.MultiIndex.from_product(
        [previous_predictions["split_id"].unique(), SVM_C_VALUES, range(3)],
        names=["split_id", "C", "inner_fold"],
    )
    observed_candidates = pd.MultiIndex.from_frame(
        previous_selection[["split_id", "C", "inner_fold"]]
    )
    if not observed_candidates.is_unique or set(observed_candidates) != set(expected_candidates):
        raise ValueError("Unvollständige oder anders konfigurierte SVM-Kandidatenauswahl.")
    for split_id, group in previous_selection.groupby("split_id"):
        means = group.groupby("C")["validation_roc_auc"].mean()
        best_c = min(means.index[means.eq(means.max())])
        assert group["selected"].eq(group["C"].eq(best_c)).all()
        assert previous_predictions.loc[previous_predictions["split_id"].eq(split_id), "best_C"].eq(best_c).all()
        assert previous_models.loc[previous_models["split_id"].eq(split_id), "best_C"].eq(best_c).all()
    # FORCE_SPLITS ist kein Weg, einen Konfigurationskonflikt zu umgehen.
    excluded = FORCE_SPLIT_IDS if RUN_TRAINING else set()
    for table, destination in [
        (previous_predictions, svm_predictions_by_split),
        (previous_selection, svm_selection_by_split),
        (previous_models, svm_models_by_split),
    ]:
        destination.append(table.loc[~table["split_id"].isin(excluded)])
    completed_split_ids = set(previous_predictions["split_id"]) - excluded
    print(f"Passender SVM-Zwischenstand: {len(completed_split_ids)} Splits.")

if RUN_TRAINING:
    for split_id in ANALYSIS_SPLIT_IDS:
        if split_id in completed_split_ids:
            continue
        best_c, threshold, candidate_results = select_svm_hyperparameters(
            split_id=split_id, cells_per_donor=TRAIN_CELLS_PER_DONOR,
        )
        split_predictions, model_parameters = fit_and_predict_outer_split(
            split_id=split_id, cells_per_donor=TRAIN_CELLS_PER_DONOR,
            best_c=best_c, threshold=threshold,
        )
        candidate_results["gate"] = GATE
        candidate_results["run_mode"] = RUN_MODE
        candidate_results["selected"] = candidate_results["C"].eq(best_c)
        svm_predictions_by_split.append(split_predictions)
        svm_selection_by_split.append(candidate_results)
        svm_models_by_split.append(model_parameters)
        SVM_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
        for parts, path in [
            (svm_predictions_by_split, SVM_PREDICTIONS_PATH),
            (svm_selection_by_split, SVM_SELECTION_PATH),
            (svm_models_by_split, SVM_MODELS_PATH),
        ]:
            pd.concat(parts, ignore_index=True).to_csv(path, index=False)
        write_run_config(CONFIG_PATH, run_config)
        print(f"Split {split_id} samt Modellparametern abgeschlossen.")

svm_predictions = pd.concat(svm_predictions_by_split, ignore_index=True)
svm_selection = pd.concat(svm_selection_by_split, ignore_index=True)
svm_models = pd.concat(svm_models_by_split, ignore_index=True)
validate_prediction_splits(svm_predictions, donor_splits)
validate_parameter_table(svm_models, svm_predictions, markers, ["split_id"], required_model_columns)
# Ein Split-Limit begrenzt die Ansicht, ohne weitere gespeicherte Splits zu löschen.
svm_predictions = svm_predictions.loc[svm_predictions["split_id"].isin(ANALYSIS_SPLIT_IDS)]
svm_selection = svm_selection.loc[svm_selection["split_id"].isin(ANALYSIS_SPLIT_IDS)]
svm_models = svm_models.loc[svm_models["split_id"].isin(ANALYSIS_SPLIT_IDS)]
assert set(svm_predictions["split_id"]) == set(ANALYSIS_SPLIT_IDS)
assert svm_predictions["train_cells_per_donor"].eq(TRAIN_CELLS_PER_DONOR).all()
svm_metrics = (
    svm_predictions.groupby("split_id")
    .apply(lambda group: pd.Series({
        "roc_auc": roc_auc_score(group["y_true"], group["score"]),
        "average_precision": average_precision_score(group["y_true"], group["score"]),
        "pr_auc": trapezoidal_pr_auc(group["y_true"], group["score"]),
        "balanced_accuracy": balanced_accuracy_score(group["y_true"], group["y_pred"]),
    }), include_groups=False).reset_index()
)
display(svm_selection)
display(svm_predictions)
display(svm_metrics)
print(f"Modellparameter: {SVM_MODELS_PATH.relative_to(PROJECT_ROOT)}")


Split 0 samt Modellparametern abgeschlossen.


Split 1 samt Modellparametern abgeschlossen.


Split 2 samt Modellparametern abgeschlossen.


Split 3 samt Modellparametern abgeschlossen.


Split 4 samt Modellparametern abgeschlossen.


...

Split 99 samt Modellparametern abgeschlossen.


,split_id,C,inner_fold,validation_roc_auc,gate,run_mode,selected
0,0,0.01,0,0.500000,gated_alive,full,True
1,0,0.01,1,0.333333,gated_alive,full,True
2,0,0.01,2,0.750000,gated_alive,full,True
3,0,0.10,0,0.500000,gated_alive,full,False
4,0,0.10,1,0.333333,gated_alive,full,False
...,...,...,...,...,...,...,...
895,99,0.10,1,0.833333,gated_alive,full,False
896,99,0.10,2,1.000000,gated_alive,full,False
897,99,1.00,0,0.500000,gated_alive,full,False
898,99,1.00,1,0.833333,gated_alive,full,False


,method,gate,run_mode,split_id,split_seed,donor_id,y_true,score,decision_threshold,y_pred,best_C,top_fraction,top_cell_count,train_cells_per_donor
0,linear_single_cell_svm,gated_alive,full,0,3003105692,a_002,1,0.934428,0.986707,0,0.01,0.01,1083,10000
1,linear_single_cell_svm,gated_alive,full,0,3003105692,a_004,0,0.473203,0.986707,0,0.01,0.01,1407,10000
2,linear_single_cell_svm,gated_alive,full,0,3003105692,a_006,0,0.730028,0.986707,0,0.01,0.01,901,10000
3,linear_single_cell_svm,gated_alive,full,0,3003105692,a_1a,0,0.700862,0.986707,0,0.01,0.01,1366,10000
4,linear_single_cell_svm,gated_alive,full,0,3003105692,a_2a,0,0.556692,0.986707,0,0.01,0.01,2370,10000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,linear_single_cell_svm,gated_alive,full,99,2699835958,a_006,0,0.955478,0.900138,1,0.01,0.01,901,10000
596,linear_single_cell_svm,gated_alive,full,99,2699835958,a_007,1,0.965099,0.900138,1,0.01,0.01,1049,10000
597,linear_single_cell_svm,gated_alive,full,99,2699835958,a_2b,0,0.648538,0.900138,0,0.01,0.01,925,10000
598,linear_single_cell_svm,gated_alive,full,99,2699835958,a_3b,0,0.639987,0.900138,0,0.01,0.01,2167,10000


,split_id,roc_auc,average_precision,pr_auc,balanced_accuracy
0,0,1.000,1.000000,1.000000,0.750
1,1,0.875,0.833333,0.791667,0.625
2,2,0.750,0.750000,0.708333,0.500
3,3,1.000,1.000000,1.000000,1.000
4,4,0.500,0.450000,0.287500,0.500
...,...,...,...,...,...
95,95,1.000,1.000000,1.000000,1.000
96,96,1.000,1.000000,1.000000,0.500
97,97,0.625,0.700000,0.662500,0.625
98,98,1.000,1.000000,1.000000,0.750


Modellparameter: results/tables/task4_svm_models_gated_alive_full.csv


## Abschluss

Die SVM ist in die gemeinsame spenderweise Auswertung integriert. Im Full-Modus werden 100 Splits auf `gated_alive` ausgewertet; jeder Split liefert sechs Testvorhersagen sowie die Gewichte, den Intercept und den Trainings-Scaler des ausgewählten Modells. Der gemeinsame Methodenvergleich und die Einordnung der Metriken folgen in `04e_comparison.ipynb`. Smoke-Ergebnisse sind ausschließlich technische Tests.
